# Fine-tune DistilRoBERTa for Proposition Detection & Relation Classification

Trains two models on UKP Argument Annotated Essays v2:
1. **Token classifier** (O/B-Prop/I-Prop) — detects argumentative proposition spans
2. **Sequence classifier** (Support/Attack/None) — pairwise relation classification

Colab-compatible with mixed precision + gradient checkpointing to fit ~4 GB VRAM.

In [ ]:
# Cell 1: Install dependencies
!pip install torch transformers scikit-learn tqdm accelerate -q

In [ ]:
# Cell 2: Check GPU
import torch
print(f"PyTorch: {torch.__version__}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name()}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: No GPU detected! Training will be slow on CPU.")

In [ ]:
# Cell 3: Mount Google Drive (skip if running locally)
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# Cell 4: Ensure data exists — download UKP dataset if needed
import os

data_dir = "/content/drive/MyDrive/reasoning_engine/data"
os.makedirs(data_dir, exist_ok=True)

required_files = [
    "ukp_tagger_train.pt", "ukp_tagger_test.pt", "label_map.json",
    "ukp_relations_train.json", "ukp_relations_test.json", "relation_map.json",
]
missing = [f for f in required_files if not os.path.exists(os.path.join(data_dir, f))]

if missing:
    print(f"Missing files: {missing}")
    print("Run prepare_ukp.py first to preprocess the UKP dataset:")
    prep_path = "/content/drive/MyDrive/reasoning_engine/scripts/prepare_ukp.py"
    print(f"  !python {prep_path} --data-dir {data_dir}")
    raise SystemExit(1)
else:
    print(f"All data files found in {data_dir}")

In [ ]:
# Cell 5: Imports and utility definitions
import json
import numpy as np
import torch
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)
from sklearn.metrics import precision_recall_fscore_support, classification_report

In [ ]:
# Cell 6: Dataset classes

class TaggerDataset(Dataset):
    """Token classification dataset (O/B-Prop/I-Prop)."""
    def __init__(self, data):
        self.data = data
    def __len__(self):
        return len(self.data)
    def __getitem__(self, idx):
        item = self.data[idx]
        return {
            "input_ids": item["input_ids"],
            "attention_mask": item["attention_mask"],
            "labels": item["labels"],
        }


class RelationDataset(Dataset):
    """Sequence pair classification dataset (Support/Attack/None)."""
    def __init__(self, pairs, tokenizer, label_map, max_length=128):
        self.encodings = []
        self.labels = []
        for pair in pairs:
            enc = tokenizer(
                pair["span1"], pair["span2"],
                truncation=True, max_length=max_length,
                padding="max_length", return_tensors="pt",
            )
            self.encodings.append({
                "input_ids": enc["input_ids"].squeeze(0),
                "attention_mask": enc["attention_mask"].squeeze(0),
            })
            self.labels.append(label_map[pair["label"]])
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, idx):
        return {
            "input_ids": self.encodings[idx]["input_ids"],
            "attention_mask": self.encodings[idx]["attention_mask"],
            "labels": torch.tensor(self.labels[idx], dtype=torch.long),
        }

In [ ]:
# Cell 7: Metric functions

def tagger_compute_metrics(p):
    predictions = p.predictions.argmax(-1)
    labels = p.label_ids
    mask = labels != -100
    predictions = predictions[mask]
    labels = labels[mask]
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, predictions, average="macro", zero_division=0
    )
    accuracy = (predictions == labels).mean()
    prop_pred = (predictions > 0).astype(int)
    prop_true = (labels > 0).astype(int)
    prop_prec, prop_rec, prop_f1, _ = precision_recall_fscore_support(
        prop_true, prop_pred, average="binary", pos_label=1, zero_division=0
    )
    return {
        "f1": f1, "precision": precision, "recall": recall, "accuracy": accuracy,
        "prop_f1": prop_f1, "prop_precision": prop_prec, "prop_recall": prop_rec,
    }


def classifier_compute_metrics(p):
    predictions = p.predictions.argmax(-1)
    labels = p.label_ids
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, predictions, average="weighted", zero_division=0
    )
    accuracy = (predictions == labels).mean()
    report = classification_report(
        labels, predictions, target_names=["None", "Support", "Attack"],
        zero_division=0, output_dict=True,
    )
    return {
        "f1": f1, "precision": precision, "recall": recall, "accuracy": accuracy,
        "attack_f1": report.get("Attack", {}).get("f1", 0),
        "support_f1": report.get("Support", {}).get("f1", 0),
    }

In [ ]:
# Cell 8: Training argument helper

def make_training_args(output_dir, batch_size=4, grad_accum=4, lr=2e-5, epochs=3):
    return TrainingArguments(
        output_dir=output_dir,
        per_device_train_batch_size=batch_size,
        gradient_accumulation_steps=grad_accum,
        gradient_checkpointing=True,
        fp16=False,
        learning_rate=lr,
        num_train_epochs=epochs,
        warmup_steps=500,
        logging_steps=10,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="f1",
        greater_is_better=True,
        report_to="none",
        save_total_limit=2,
    )

In [ ]:
# Cell 9: Train Proposition Detector (Token Classifier)

model_name = "distilroberta-base"
output_base = "/content/drive/MyDrive/reasoning_engine/models"
os.makedirs(output_base, exist_ok=True)

print("=" * 60)
print("Training Proposition Detector (Token Classifier)")
print("=" * 60)

tagger_model = AutoModelForTokenClassification.from_pretrained(
    model_name,
    num_labels=3,
    id2label={0: "O", 1: "B-Prop", 2: "I-Prop"},
    label2id={"O": 0, "B-Prop": 1, "I-Prop": 2},
)

tagger_out = os.path.join(output_base, "roberta-proposition-detector")
train_tagger = torch.load(os.path.join(data_dir, "ukp_tagger_train.pt"))
test_tagger = torch.load(os.path.join(data_dir, "ukp_tagger_test.pt"))

tagger_args = make_training_args(output_dir=tagger_out)

tagger_trainer = Trainer(
    model=tagger_model,
    args=tagger_args,
    train_dataset=TaggerDataset(train_tagger),
    eval_dataset=TaggerDataset(test_tagger),
    compute_metrics=tagger_compute_metrics,
)

tagger_trainer.train()
tagger_trainer.save_model(tagger_out)
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.save_pretrained(tagger_out)
print(f"Proposition Detector saved to {tagger_out}")

In [ ]:
# Cell 10: Train Relation Classifier (Sequence Pair Classifier)

print("=" * 60)
print("Training Relation Classifier (Sequence Pair Classifier)")
print("=" * 60)

with open(os.path.join(data_dir, "relation_map.json")) as f:
    relation_map = json.load(f)

classifier_model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=3,
    id2label={0: "None", 1: "Support", 2: "Attack"},
    label2id={"None": 0, "Support": 1, "Attack": 2},
)

classifier_out = os.path.join(output_base, "roberta-relation-classifier")
tokenizer = AutoTokenizer.from_pretrained(model_name)

with open(os.path.join(data_dir, "ukp_relations_train.json")) as f:
    train_relations = json.load(f)
with open(os.path.join(data_dir, "ukp_relations_test.json")) as f:
    test_relations = json.load(f)

classifier_args = make_training_args(output_dir=classifier_out, batch_size=8)

classifier_trainer = Trainer(
    model=classifier_model,
    args=classifier_args,
    train_dataset=RelationDataset(train_relations, tokenizer, relation_map),
    eval_dataset=RelationDataset(test_relations, tokenizer, relation_map),
    compute_metrics=classifier_compute_metrics,
)

classifier_trainer.train()
classifier_trainer.save_model(classifier_out)
tokenizer.save_pretrained(classifier_out)
print(f"Relation Classifier saved to {classifier_out}")

In [ ]:
# Cell 11: Done
print("Both models trained successfully!")
print(f"  Proposition Detector: {tagger_out}/")
print(f"  Relation Classifier:  {classifier_out}/")